# SDFVD 2.0 source-pair cross-validation (Colab)

This notebook carries the current source bundle and refreshes its base64 copy in your selected Google Drive folder when setup runs. It extracts the SDFVD 2.0 archive from Drive to temporary Colab storage, generates labels, groups each inferred real/fake source pair with its augmentations, prepares cached features, and (in the next cell) can run five-fold paired-source-disjoint training. Each fold uses separate validation groups for checkpoint selection and threshold calibration. Drive output contains fold metrics, original-only diagnostics, source-pair score summaries, split manifests, and fold checkpoints. The full derived feature cache stays in temporary Colab storage.

The metadata builder uses the Mendeley CC BY 4.0 record as the dataset-level usage basis and preserves attribution; rights to underlying source media are not independently verified. It refuses ambiguous labels and records naming/count anomalies. Fold averages estimate performance within this dataset; external reliability requires a separate dataset with verified source identities.


In [ ]:
from google.colab import auth, drive
from pathlib import Path
if not Path('/content/drive/MyDrive').is_dir():
    drive.mount('/content/drive')
else:
    print('Google Drive is already mounted; reusing the current mount.')
auth.authenticate_user()
import base64, json, shutil, subprocess, sys, zipfile
from google.auth import default
from googleapiclient.discovery import build
from datetime import datetime, timezone

drive_credentials, _ = default()
drive_api = build('drive', 'v3', credentials=drive_credentials)
PROJECT_FOLDER_ID = 'YOUR_GOOGLE_DRIVE_FOLDER_ID'
folder_names = []
current_folder_id = PROJECT_FOLDER_ID
seen_folder_ids = set()
while current_folder_id and current_folder_id != 'root' and current_folder_id not in seen_folder_ids:
    seen_folder_ids.add(current_folder_id)
    info = drive_api.files().get(fileId=current_folder_id, fields='id,name,mimeType,parents').execute()
    if info.get('mimeType') != 'application/vnd.google-apps.folder':
        raise ValueError(f"Expected Drive folder, got {info.get('mimeType')} for {current_folder_id}")
    if info.get('name') == 'My Drive':
        break
    folder_names.append(info['name'])
    parents = info.get('parents', [])
    current_folder_id = parents[0] if parents else None
DRIVE_PROJECT_FOLDER = Path('/content/drive/MyDrive').joinpath(*reversed(folder_names))
if not DRIVE_PROJECT_FOLDER.is_dir():
    raise FileNotFoundError(f'Could not resolve the provided Drive folder ID to a mounted MyDrive path: {DRIVE_PROJECT_FOLDER}')
SOURCE_B64 = DRIVE_PROJECT_FOLDER / 'poor_guys_deepfake_colab_source.b64'
SOURCE_BUNDLE_B64 = (
    '...'
)
SOURCE_B64.write_text(SOURCE_BUNDLE_B64, encoding='ascii')
PROJECT_ZIP.write_bytes(base64.b64decode(SOURCE_B64.read_text(encoding='ascii').strip()))
SOURCE_B64.write_text(SOURCE_BUNDLE_B64, encoding='ascii')
PROJECT_ZIP.write_bytes(base64.b64decode(SOURCE_B64.read_text(encoding='ascii').strip()))
PROJECT_ZIP = Path('/content/poor_guys_deepfake_colab_source.zip')
ARCHIVE_ZIP = DRIVE_PROJECT_FOLDER / 'SDFVD2.0_v1.zip'
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
PROJECT_ROOT = Path('/content') / f'sdfvd2_source_{RUN_ID}' / 'poor_guys_deepfake_project'
EXTRACT_ROOT = PROJECT_ROOT.parent
media_candidates = sorted(Path('/content').glob('sdfvd2_media_*'),
                          key=lambda path: path.stat().st_mtime, reverse=True)
MEDIA_ROOT = None
for candidate in media_candidates:
    if candidate.is_dir() and sum(1 for _ in candidate.rglob('*.mp4')) == 927:
        MEDIA_ROOT = candidate
        break
REUSE_EXTRACTED_MEDIA = MEDIA_ROOT is not None
if MEDIA_ROOT is None:
    MEDIA_ROOT = Path('/content') / f'sdfvd2_media_{RUN_ID}'
WORK_OUTPUT = Path('/content') / f'sdfvd2_run_{RUN_ID}'
PERSIST_OUTPUT = DRIVE_PROJECT_FOLDER / f'sdfvd2_training_{RUN_ID}'
METADATA_CSV = WORK_OUTPUT / 'sdfvd2_metadata.csv'
METADATA_REPORT = WORK_OUTPUT / 'sdfvd2_metadata_report.json'
DATASET_CARD = WORK_OUTPUT / 'sdfvd2_dataset_card.md'

if not ARCHIVE_ZIP.is_file():
    raise FileNotFoundError(f'Required Drive dataset archive is missing: {ARCHIVE_ZIP}')
EXTRACT_ROOT.mkdir(parents=True)
MEDIA_ROOT.mkdir(parents=True, exist_ok=True)
WORK_OUTPUT.mkdir(parents=True)
with zipfile.ZipFile(PROJECT_ZIP) as bundle:
    bundle.extractall(EXTRACT_ROOT)
if not (PROJECT_ROOT / 'multimodal' / 'run_pipeline.py').is_file():
    raise FileNotFoundError('Source bundle has the wrong internal layout; upload the generated ZIP.')

if REUSE_EXTRACTED_MEDIA:
    print(f'Reusing existing extracted media at {MEDIA_ROOT}; skipping archive extraction.', flush=True)
else:
    MEDIA_ROOT.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(ARCHIVE_ZIP) as archive:
        members = archive.infolist()
        for number, member in enumerate(members, 1):
            destination = (MEDIA_ROOT / member.filename).resolve()
            if destination != MEDIA_ROOT and MEDIA_ROOT not in destination.parents:
                raise ValueError(f'Unsafe archive member path: {member.filename}')
            if member.is_dir():
                destination.mkdir(parents=True, exist_ok=True)
                continue
            destination.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(member) as source, destination.open('wb') as target:
                shutil.copyfileobj(source, target)
            if number % 100 == 0 or number == len(members):
                print(f'Extracted and CRC-checked {number}/{len(members)} archive entries', flush=True)

subprocess.run([sys.executable, str(PROJECT_ROOT / 'data_ingestion' / 'prepare_sdfvd2_metadata.py'),
                '--media-root', str(MEDIA_ROOT), '--output', str(METADATA_CSV),
                '--report', str(METADATA_REPORT), '--dataset-card', str(DATASET_CARD)],
               cwd=PROJECT_ROOT, check=True)
metadata_summary = json.loads(METADATA_REPORT.read_text(encoding='utf-8'))
print(json.dumps({k: metadata_summary[k] for k in ('videos_found', 'label_counts',
      'unique_candidate_source_groups', 'paired_source_group_count', 'source_group_counts_by_label', 'group_rule_counts',
      'vid_number_alias_count', 'augmented_filename_count', 'ambiguous_label_count',
      'count_difference_from_published')}, indent=2))
if metadata_summary['videos_found'] != 927 or metadata_summary['label_counts'] != {'real': 456, 'synthetic': 471}:
    raise RuntimeError('Archive inventory differs from the checksum-verified SDFVD2 inventory (927; 456 real/471 synthetic). Stop and inspect the generated report.')
if metadata_summary['ambiguous_label_count']:
    raise RuntimeError('Ambiguous label evidence found. Review the generated metadata/report before training; nothing will be silently relabeled.')
if metadata_summary.get('source_group_counts_by_label') != {'real': 53, 'synthetic': 53} or metadata_summary.get('unique_candidate_source_groups') != 53:
    raise RuntimeError('Expected 53 paired real/fake source groups (each represented in both labels). Stop and inspect the group audit before training.')
import torch
print('GPU available:', torch.cuda.is_available())
print('Inputs and dataset inventory validated. Run the next cell to start training.')


## Run source-pair cross-validation

Run this cell after preparation completes. It reuses the extracted media and cached features when available, then trains five source-pair-disjoint folds. This takes roughly five model-training passes. It reports fold means and variability; it does not promise a high score or create a single deployment checkpoint.


In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import base64, json, subprocess, sys, zipfile
from uuid import uuid4

_setup_names = ('SOURCE_B64', 'MEDIA_ROOT', 'METADATA_CSV', 'METADATA_REPORT',
                'DATASET_CARD', 'WORK_OUTPUT', 'DRIVE_PROJECT_FOLDER')
_missing_setup = [name for name in _setup_names if name not in globals()]
if _missing_setup:
    raise RuntimeError('Setup cell state is missing: ' + ', '.join(_missing_setup) +
                       '. Run the first notebook setup cell in this Colab runtime, then rerun this cell.')

if not Path(MEDIA_ROOT).is_dir():
    raise RuntimeError(f'Extracted dataset is missing at {MEDIA_ROOT}. Run the first notebook setup cell in this runtime.')
if not Path(SOURCE_B64).is_file():
    raise FileNotFoundError(f'Updated project bundle is missing: {SOURCE_B64}. Run the first notebook setup cell.')

CACHED_PIPELINE_OUTPUT = globals().get('CACHED_PIPELINE_OUTPUT')
if CACHED_PIPELINE_OUTPUT is None:
    CACHED_PIPELINE_OUTPUT = globals().get('PREVIOUS_PIPELINE_OUTPUT') or globals().get('PIPELINE_OUTPUT')
if CACHED_PIPELINE_OUTPUT is None:
    cache_candidates = [path for path in Path('/content').glob('sdfvd2_run_*/pipeline_*')
                        if (path / 'manifest.jsonl').is_file() and
                        (path / 'derived_av' / 'av_manifest.jsonl').is_file()]
    if cache_candidates:
        CACHED_PIPELINE_OUTPUT = str(max(cache_candidates, key=lambda path: path.stat().st_mtime))
        print(f'Reusing detected AV feature cache: {CACHED_PIPELINE_OUTPUT}')
ATTEMPT_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ') + '_' + uuid4().hex[:6]
PROJECT_ZIP = Path('/content') / f'poor_guys_deepfake_colab_source_{ATTEMPT_ID}.zip'
EXTRACT_ROOT = Path('/content') / f'sdfvd2_source_{ATTEMPT_ID}'
PROJECT_ROOT = EXTRACT_ROOT / 'poor_guys_deepfake_project'
PROJECT_ZIP.write_bytes(base64.b64decode(SOURCE_B64.read_text(encoding='ascii').strip()))
EXTRACT_ROOT.mkdir(parents=True, exist_ok=False)
with zipfile.ZipFile(PROJECT_ZIP) as bundle:
    bundle.extractall(EXTRACT_ROOT)
if not (PROJECT_ROOT / 'splits' / 'make_splits.py').is_file():
    raise FileNotFoundError('Updated source bundle is missing the group-disjoint split implementation.')
subprocess.run([sys.executable, str(PROJECT_ROOT / 'data_ingestion' / 'prepare_sdfvd2_metadata.py'),
                '--media-root', str(MEDIA_ROOT), '--output', str(METADATA_CSV),
                '--report', str(METADATA_REPORT), '--dataset-card', str(DATASET_CARD)],
               cwd=PROJECT_ROOT, check=True)
metadata_summary = json.loads(METADATA_REPORT.read_text(encoding='utf-8'))
print(json.dumps({k: metadata_summary[k] for k in ('videos_found', 'label_counts',
      'unique_candidate_source_groups', 'paired_source_group_count', 'source_group_counts_by_label', 'group_rule_counts',
      'vid_number_alias_count', 'ambiguous_label_count',
      'count_difference_from_published')}, indent=2))
if metadata_summary['videos_found'] != 927 or metadata_summary['label_counts'] != {'real': 456, 'synthetic': 471}:
    raise RuntimeError('Archive inventory differs from the checksum-verified SDFVD2 inventory (927; 456 real/471 synthetic). Stop and inspect the generated report.')
if metadata_summary['ambiguous_label_count']:
    raise RuntimeError('Ambiguous label evidence found. Review the generated metadata/report before training; nothing will be silently relabeled.')
if metadata_summary.get('source_group_counts_by_label') != {'real': 53, 'synthetic': 53} or metadata_summary.get('unique_candidate_source_groups') != 53:
    raise RuntimeError('Expected 53 paired real/fake source groups (each represented in both labels). Stop and inspect the group audit before training.')
PIPELINE_OUTPUT = WORK_OUTPUT / f'pipeline_{ATTEMPT_ID}'
PERSIST_OUTPUT = DRIVE_PROJECT_FOLDER / f'sdfvd2_training_{ATTEMPT_ID}'

cmd = [sys.executable, str(PROJECT_ROOT / 'multimodal' / 'run_pipeline.py'),
       '--media-root', str(MEDIA_ROOT), '--metadata', str(METADATA_CSV),
       '--dataset-card', str(DATASET_CARD), '--output-dir', str(PIPELINE_OUTPUT),
       '--epochs', '10', '--batch-size', '16', '--workers', '2', '--device', 'auto',
       '--split-seed', '2026', '--train-fraction', '0.70', '--validation-fraction', '0.15',
       '--group-cv-folds', '5',
       '--pretrained-visual']
if CACHED_PIPELINE_OUTPUT and (Path(CACHED_PIPELINE_OUTPUT) / 'manifest.jsonl').is_file() and \
        (Path(CACHED_PIPELINE_OUTPUT) / 'derived_av' / 'av_manifest.jsonl').is_file():
    cmd.extend(['--reuse-preprocessed-from', str(CACHED_PIPELINE_OUTPUT)])
    print('Revalidating current media/labels against cached features, then reusing extraction if they match.')
print('Starting SDFVD 2.0 preprocessing, group-disjoint training, validation calibration, and held-out evaluation...')
process = subprocess.Popen(cmd, cwd=PROJECT_ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                           text=True, bufsize=1)
for line in process.stdout:
    print(line, end='')
return_code = process.wait()
if return_code:
    raise RuntimeError(f'Pipeline stopped with exit code {return_code}; see the final stage output above.')
print('Training and held-out evaluation completed:', PIPELINE_OUTPUT)
if (PIPELINE_OUTPUT / 'derived_av' / 'av_manifest.jsonl').is_file():
    CACHED_PIPELINE_OUTPUT = PIPELINE_OUTPUT


## Save and display the results

This copies fold-wise metrics, source-group diagnostics, thresholds, predictions, split manifests, and metadata audit to a run-specific folder in Drive. Each fold has its own checkpoint. Scores from different fold models are summarized per fold rather than pooled. The large derived feature cache stays in temporary Colab storage.


In [ ]:
cv_report = PIPELINE_OUTPUT / 'group_cv' / 'group_cv_summary.json'
if not cv_report.is_file():
    raise FileNotFoundError('Group cross-validation summary is missing; pipeline did not finish all folds.')
PERSIST_OUTPUT.mkdir(parents=True, exist_ok=True)
for name in ('run_config.json',):
    src = PIPELINE_OUTPUT / name
    if src.is_file(): shutil.copy2(src, PERSIST_OUTPUT / name)
for name in ('sdfvd2_metadata.csv', 'sdfvd2_metadata_report.json', 'sdfvd2_dataset_card.md'):
    src = WORK_OUTPUT / name
    if src.is_file(): shutil.copy2(src, PERSIST_OUTPUT / name)
shutil.copytree(PIPELINE_OUTPUT / 'group_cv', PERSIST_OUTPUT / 'group_cv', dirs_exist_ok=True)
summary = json.loads(cv_report.read_text(encoding='utf-8'))
print('SDFVD 2.0 five-fold paired-source cross-validation')
for branch, report in summary['branches'].items():
    print(f'\n{branch} fold-mean metrics')
    print(json.dumps(report['summary'], indent=2))
    print('Original-only fold summary:', json.dumps(report['original_only_summary'], indent=2))
    print('Per-fold source-pair-balanced AUC:', report['paired_source_group_auc_by_fold'])
print('Drive results folder:', PERSIST_OUTPUT)
print('Fold-specific checkpoints are under:', PERSIST_OUTPUT / 'group_cv')
print('This run estimates within-dataset source-pair performance; it does not establish external generalization.')


## Interpretation limits

- The 927-file archive count differs by five from the 922 reported by the dataset record. The notebook asserts the previously CRC-checked inventory and records the discrepancy rather than dropping files.
- The SDFVD source-family convention pairs real `vN` with manipulated `vsN`; each pair and every augmentation descendant stays in one fold. This prevents paired-source leakage, but does not prove broader subject/session identity separation.
- The Mendeley record identifies CC BY 4.0. The dataset card and metrics retain attribution; rights to underlying source videos were not independently verified, so do not redistribute the media or model without checking applicable terms.
- The notebook uses five-fold source-pair cross-validation. Each fold has a validation subset for checkpoint selection and threshold calibration; fold metrics remain separate because model scores may not share one calibration scale.
- Audio-only evaluation is not reported when no audio is decodable. The SDFVD 2.0 run so far had no audio in any split, so results are video-only.
- Small independent source counts limit precision even when there are many augmented files. External reliability requires evaluation on a separate dataset with verified source identities.
